# FruitScan · Entrenar los modelos de mango en Colab (GPU)

Respaldo para entrenar con GPU los mismos scripts de `ml/scripts/`, sin cambiar código: solo cambian los argumentos.

**Antes de correrlo**
1. En Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*.
2. En tu Google Drive crea la carpeta `FruitScan/` y sube los dos zips que generan los scripts de preparación en tu computadora:
   - `ml/data/processed/clasificador_mango.zip` (sale de `preparar_clasificador.py`)
   - `ml/data/processed/detector_mango.zip` (sale de `preparar_detector.py`)
3. La rama de la celda siguiente debe estar subida a GitHub (`git push`).

Después usa *Entorno de ejecución → Ejecutar todas*. Los pesos (`best.pt`) y las métricas quedan en `FruitScan/modelos/` de tu Drive.

In [ ]:
REPO_URL = "https://github.com/JazielAguirre/Hackaton2026.git"
RAMA = "entrenamiento-mango"  # cámbiala a "main" cuando la rama se fusione
CARPETA_DRIVE = "/content/drive/MyDrive/FruitScan"
REPO = "/content/repo"

## 1. Revisar la GPU y montar Drive

In [ ]:
!nvidia-smi -L

from google.colab import drive
drive.mount("/content/drive")

## 2. Bajar el código e instalar dependencias

Colab ya trae `torch` y `torchvision` compilados para su GPU, así que se instala el resto de `ml/requirements.txt` con las versiones fijas. Si pip avisa de conflictos con paquetes propios de Colab, no afectan al entrenamiento.

In [ ]:
!rm -rf {REPO}
!git clone --depth 1 --branch {RAMA} {REPO_URL} {REPO}
!grep -vE "^(torch|torchvision)==" {REPO}/ml/requirements.txt > /tmp/requirements_colab.txt
!pip install -q -r /tmp/requirements_colab.txt

## 3. Descomprimir los datos procesados

In [ ]:
!mkdir -p {REPO}/ml/data/processed
!unzip -q -o "{CARPETA_DRIVE}/clasificador_mango.zip" -d {REPO}/ml/data/processed
!unzip -q -o "{CARPETA_DRIVE}/detector_mango.zip" -d {REPO}/ml/data/processed
!ls {REPO}/ml/data/processed/clasificador/mango {REPO}/ml/data/processed/detector/mango

## 4. Guardar resultados en Drive

Se llama después de cada entrenamiento, así no se pierde nada si Colab se desconecta a la mitad.

In [ ]:
import json
import shutil
from pathlib import Path

def guardar_en_drive(nombre: str) -> None:
    """Copia best.pt y metricas.json de ml/modelos/<nombre>/ a Drive, más las gráficas del entrenamiento."""
    origen = Path(REPO) / "ml" / "modelos" / nombre
    destino = Path(CARPETA_DRIVE) / "modelos" / nombre
    shutil.copytree(origen, destino, dirs_exist_ok=True)
    shutil.make_archive(str(destino / "runs"), "zip", Path(REPO) / "ml" / "runs")
    print(json.dumps(json.loads((origen / "metricas.json").read_text()), indent=2, ensure_ascii=False))
    print(f"Guardado en {destino}")

## 5. Clasificador

Con GPU se dan más épocas y un batch más grande que en CPU. Para repetir exactamente el entrenamiento de CPU, quita `--epochs`, `--patience` y `--batch`.

In [ ]:
%cd {REPO}
!python ml/scripts/entrenar_clasificador.py --device 0 --batch 64 --epochs 50 --patience 10
guardar_en_drive("clasificador_mango")

## 6. Detector

En GPU se entrena con fotos de 640 px, sin congelar capas y con más épocas. La caché en RAM se decide sola (`--cache auto`).

In [ ]:
%cd {REPO}
!python ml/scripts/entrenar_detector.py --device 0 --imgsz 640 --batch 32 --epochs 100 --patience 20 --freeze 0
guardar_en_drive("detector")